# Chapter 39: Authentication & Security — Colab Notebook

This notebook actually runs the FastAPI/bcrypt/PyJWT code that the lesson page (`ch39-authentication-security.html`) shows as reference-only. FastAPI needs a real ASGI runtime to execute, which Pyodide (the in-browser Python powering the rest of the course) cannot provide — no real network sockets in a WebAssembly sandbox.

Every cell below uses `fastapi.testclient.TestClient`, which drives the app **in-process** (no real socket, no port) — deterministic and reproducible every time you run this notebook, in Colab or anywhere else.

Run the cells top to bottom. Install dependencies first if needed:

```
!pip install -q fastapi uvicorn httpx bcrypt pyjwt python-multipart
```


## 39.2 — Hashing and Verifying a Password

In [1]:
import bcrypt

def hash_password(plain: str) -> str:
    # bcrypt works on bytes; store the hash as a str so it fits a DB text column / JSON
    return bcrypt.hashpw(plain.encode(), bcrypt.gensalt()).decode()

def verify_password(plain: str, hashed: str) -> bool:
    return bcrypt.checkpw(plain.encode(), hashed.encode())

hashed = hash_password("s3cret-password")
print("hash starts with:", hashed[:7])
print("stored as:", type(hashed).__name__)

print("correct password verifies:", verify_password("s3cret-password", hashed))
print("wrong password verifies:", verify_password("wrong-guess", hashed))

# same password, different hash every time (random salt is embedded in the hash)
print("two hashes of one password differ:", hash_password("s3cret-password") != hashed)

# bcrypt only uses the first 72 bytes; current releases refuse longer input instead of truncating
try:
    hash_password("x" * 73)
except ValueError as e:
    print("73-byte password rejected:", e)


hash starts with: $2b$12$
stored as: str
correct password verifies: True


wrong password verifies: False
two hashes of one password differ: True
73-byte password rejected: password cannot be longer than 72 bytes, truncate manually if necessary (e.g. my_password[:72])


## 39.3 — OAuth2PasswordBearer, fake_users_db, and the /token Route

In [2]:
from fastapi import FastAPI, Depends, HTTPException, status
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm
from fastapi.testclient import TestClient

app = FastAPI(title="Candidate Scoring API")
oauth2_scheme = OAuth2PasswordBearer(tokenUrl="token")

fake_users_db = {
    "recruiter1": {"username": "recruiter1", "hashed_password": hash_password("hunter2")}
}

CANDIDATES = {
    1: {"name": "Alice", "score": 0.92},
    2: {"name": "Bob",   "score": 0.75},
    3: {"name": "Carol", "score": 0.81},
    4: {"name": "Dave",  "score": 0.60},
    5: {"name": "Eve",   "score": 0.55},
}


## 39.4 — Creating and Verifying JWTs

In [3]:
import jwt
from datetime import datetime, timedelta, timezone

SECRET_KEY = "dev-only-secret-change-in-production"
ALGORITHM = "HS256"

def create_access_token(username: str) -> str:
    payload = {"sub": username, "exp": datetime.now(timezone.utc) + timedelta(minutes=30)}
    return jwt.encode(payload, SECRET_KEY, algorithm=ALGORITHM)

token = create_access_token("recruiter1")
print("token type:", type(token).__name__)
print("token (truncated):", token[:40] + "...")
print("segments (header.payload.signature):", len(token.split(".")))

decoded = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
print("decoded sub:", decoded["sub"])

# an expired token is rejected; ExpiredSignatureError is a subclass of InvalidTokenError
expired = jwt.encode(
    {"sub": "recruiter1", "exp": datetime.now(timezone.utc) - timedelta(seconds=1)},
    SECRET_KEY, algorithm=ALGORITHM,
)
try:
    jwt.decode(expired, SECRET_KEY, algorithms=[ALGORITHM])
except jwt.InvalidTokenError as e:
    print("expired token rejected:", type(e).__name__, "-", e)
print("ExpiredSignatureError is an InvalidTokenError:", issubclass(jwt.ExpiredSignatureError, jwt.InvalidTokenError))

# a tampered signature is rejected too
head, body, sig = token.split(".")
tampered = ".".join([head, body, sig[:-2] + ("AA" if not sig.endswith("AA") else "BB")])
try:
    jwt.decode(tampered, SECRET_KEY, algorithms=[ALGORITHM])
except jwt.InvalidTokenError as e:
    print("tampered token rejected:", type(e).__name__)


token type: str
token (truncated): eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.eyJ...
segments (header.payload.signature): 3
decoded sub: recruiter1
expired token rejected: ExpiredSignatureError - Signature has expired
ExpiredSignatureError is an InvalidTokenError: True
tampered token rejected: InvalidSignatureError


## 39.5 — The get_current_user Dependency

In [4]:
def get_current_user(token: str = Depends(oauth2_scheme)):
    try:
        payload = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
        username = payload.get("sub")
        if username is None:
            raise HTTPException(status_code=status.HTTP_401_UNAUTHORIZED, detail="Invalid token")
        return username
    except jwt.InvalidTokenError:
        raise HTTPException(status_code=status.HTTP_401_UNAUTHORIZED, detail="Invalid token")

@app.post("/token")
def login(form_data: OAuth2PasswordRequestForm = Depends()):
    user = fake_users_db.get(form_data.username)
    if not user or not verify_password(form_data.password, user["hashed_password"]):
        raise HTTPException(status_code=401, detail="Incorrect username or password")
    return {"access_token": create_access_token(user["username"]), "token_type": "bearer"}


## 39.6 — Protecting the Write Endpoint

In [5]:
from pydantic import BaseModel, Field

class CandidateIn(BaseModel):
    name: str
    score: float = Field(gt=0, le=1)

class CandidateOut(BaseModel):
    id: int
    name: str
    score: float

@app.get("/candidates/{candidate_id}", response_model=CandidateOut)
def get_candidate(candidate_id: int):
    if candidate_id not in CANDIDATES:
        raise HTTPException(status_code=404, detail="Candidate not found")
    return {"id": candidate_id, **CANDIDATES[candidate_id]}

@app.post("/candidates", response_model=CandidateOut, status_code=201)
def create_candidate(candidate: CandidateIn, current_user: str = Depends(get_current_user)):
    new_id = max(CANDIDATES) + 1
    CANDIDATES[new_id] = candidate.model_dump()
    return {"id": new_id, **candidate.model_dump()}


## 39.7 — Testing With TestClient: Login, Then a Protected Call

In [6]:
client = TestClient(app)

login_response = client.post("/token", data={"username": "recruiter1", "password": "hunter2"})
print("login status:", login_response.status_code)
token = login_response.json()["access_token"]

authorized_response = client.post(
    "/candidates",
    json={"name": "Frank", "score": 0.70},
    headers={"Authorization": f"Bearer {token}"}
)
print("authorized POST status:", authorized_response.status_code)
print("authorized POST body:", authorized_response.json())

unauthorized_response = client.post("/candidates", json={"name": "Ghost", "score": 0.5})
print("unauthorized POST status:", unauthorized_response.status_code)

bad_login = client.post("/token", data={"username": "recruiter1", "password": "wrong"})
print("bad login status:", bad_login.status_code)


login status: 200
authorized POST status: 201
authorized POST body: {'id': 6, 'name': 'Frank', 'score': 0.7}
unauthorized POST status: 401
bad login status: 401


## Mini Project: Login & Protected Writes

A fresh, self-contained app combining every piece from this chapter: a `/token` login route, JWT issuance, and a `get_current_user` dependency protecting the create-candidate route — proven with TestClient, including the unauthorized path.

In [7]:
from fastapi import FastAPI, Depends, HTTPException, status
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field
import jwt
from datetime import datetime, timedelta, timezone

project_app = FastAPI(title="Candidate Scoring API — Mini Project")
project_oauth2_scheme = OAuth2PasswordBearer(tokenUrl="token")

PROJECT_SECRET_KEY = "dev-only-secret-change-in-production"
PROJECT_ALGORITHM = "HS256"

project_users_db = {
    "recruiter1": {"username": "recruiter1", "hashed_password": hash_password("hunter2")}
}
PROJECT_CANDIDATES = {
    1: {"name": "Alice", "score": 0.92},
    2: {"name": "Bob",   "score": 0.75},
}

class ProjectCandidateIn(BaseModel):
    name: str
    score: float = Field(gt=0, le=1)

class ProjectCandidateOut(BaseModel):
    id: int
    name: str
    score: float

def project_create_token(username: str) -> str:
    payload = {"sub": username, "exp": datetime.now(timezone.utc) + timedelta(minutes=30)}
    return jwt.encode(payload, PROJECT_SECRET_KEY, algorithm=PROJECT_ALGORITHM)

def project_get_current_user(token: str = Depends(project_oauth2_scheme)):
    try:
        payload = jwt.decode(token, PROJECT_SECRET_KEY, algorithms=[PROJECT_ALGORITHM])
        username = payload.get("sub")
        if username is None:
            raise HTTPException(status_code=status.HTTP_401_UNAUTHORIZED, detail="Invalid token")
        return username
    except jwt.InvalidTokenError:
        raise HTTPException(status_code=status.HTTP_401_UNAUTHORIZED, detail="Invalid token")

@project_app.post("/token")
def project_login(form_data: OAuth2PasswordRequestForm = Depends()):
    user = project_users_db.get(form_data.username)
    if not user or not verify_password(form_data.password, user["hashed_password"]):
        raise HTTPException(status_code=401, detail="Incorrect username or password")
    return {"access_token": project_create_token(user["username"]), "token_type": "bearer"}

@project_app.post("/candidates", response_model=ProjectCandidateOut, status_code=201)
def project_create_candidate(candidate: ProjectCandidateIn, current_user: str = Depends(project_get_current_user)):
    new_id = max(PROJECT_CANDIDATES) + 1
    PROJECT_CANDIDATES[new_id] = candidate.model_dump()
    return {"id": new_id, **candidate.model_dump()}

client = TestClient(project_app)

login_resp = client.post("/token", data={"username": "recruiter1", "password": "hunter2"})
assert login_resp.status_code == 200
token = login_resp.json()["access_token"]

good_resp = client.post("/candidates", json={"name": "Grace", "score": 0.88}, headers={"Authorization": f"Bearer {token}"})
assert good_resp.status_code == 201

no_token_resp = client.post("/candidates", json={"name": "Ghost", "score": 0.5})
assert no_token_resp.status_code == 401

bad_login_resp = client.post("/token", data={"username": "recruiter1", "password": "wrong"})
assert bad_login_resp.status_code == 401

print("Project checklist: PASSED (login issues JWT, protected POST accepts valid token, rejects missing token, rejects bad login)")


Project checklist: PASSED (login issues JWT, protected POST accepts valid token, rejects missing token, rejects bad login)


### Next: Chapter 40 — Testing, Middleware & Background Tasks (Colab)